In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')



In [2]:
# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_rows', 100)

# Set plot style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

In [17]:

retail_rice_raw = pd.read_csv('../unclean_data/Retail_CAN_RICE.csv', 
                               header=None, 
                               encoding='utf-8-sig',
                               on_bad_lines='warn',  # Warn but continue
                               engine='python')  # Python engine is more flexible

# Load food CPI normally
food_cpi_df = pd.read_csv('../unclean_data/CPI_CAN_RICE.csv')

print("="*80)
print("RAW DATA LOADED - CANADA")
print("="*80)
print(f"\n1. Canada Retail Rice Prices (raw):")
print(f"   Shape: {retail_rice_raw.shape}")
print(f"   Total rows: {len(retail_rice_raw)}")
print(f"\n   Checking key rows:")
print(f"   Row 9 (dates): {retail_rice_raw.iloc[9, 0:5].tolist() if len(retail_rice_raw) > 9 else 'Not found'}")
print(f"   Row 11 (brown rice): {retail_rice_raw.iloc[11, 0:5].tolist() if len(retail_rice_raw) > 11 else 'Not found'}")
print(f"   Row 12 (white rice): {retail_rice_raw.iloc[12, 0:5].tolist() if len(retail_rice_raw) > 12 else 'Not found'}")

print(f"\n2. Canada Food CPI:")
print(f"   Shape: {food_cpi_df.shape}")
print(f"   Date Range: {food_cpi_df['observation_date'].min()} to {food_cpi_df['observation_date'].max()}")
print(f"\n   First 5 rows:")
print(food_cpi_df.head())

RAW DATA LOADED - CANADA

1. Canada Retail Rice Prices (raw):
   Shape: (8, 1)
   Total rows: 8

   Checking key rows:
   Row 9 (dates): Not found
   Row 11 (brown rice): Not found
   Row 12 (white rice): Not found

2. Canada Food CPI:
   Shape: (112, 2)
   Date Range: 2015-12-01 to 2025-03-01

   First 5 rows:
  observation_date  CANCP010000CTGYM
0       2015-12-01          0.475448
1       2016-01-01          0.544243
2       2016-02-01          0.522611
3       2016-03-01          0.468672
4       2016-04-01          0.385286


In [20]:
# ============================================================================
# CELL 3: Clean Rice Retail Prices - Transform from Wide to Long Format
# ============================================================================
print("="*80)
print("TRANSFORMING RETAIL RICE DATA: WIDE → LONG FORMAT")
print("="*80)

# First, let's check what rows we actually have
print(f"Total rows in retail_rice_raw: {len(retail_rice_raw)}")
print(f"Total columns: {len(retail_rice_raw.columns)}")

# Let's look at the first column of each row to find our data
print("\nSearching for rice data rows:")
for i in range(min(15, len(retail_rice_raw))):
    first_val = retail_rice_raw.iloc[i, 0]
    if pd.notna(first_val):
        print(f"Row {i}: {first_val}")

# Based on the file structure, find the rows
# Usually: Products row, then Brown rice, then White rice
products_row = None
brown_row = None
white_row = None

for i in range(len(retail_rice_raw)):
    val = str(retail_rice_raw.iloc[i, 0]).strip().lower()
    if 'products' in val or 'january 2017' in str(retail_rice_raw.iloc[i, 1]).lower():
        products_row = i
        print(f"\n✓ Found date header row at index {i}")
    elif 'brown rice' in val:
        brown_row = i
        print(f"✓ Found brown rice at index {i}: {retail_rice_raw.iloc[i, 0]}")
    elif 'white rice' in val:
        white_row = i
        print(f"✓ Found white rice at index {i}: {retail_rice_raw.iloc[i, 0]}")

# Check if we found everything
if products_row is None or brown_row is None or white_row is None:
    print(f"\n❌ ERROR: Could not find all required rows")
    print(f"Products row: {products_row}")
    print(f"Brown row: {brown_row}")
    print(f"White row: {white_row}")
    raise ValueError("Missing required data rows")

# Extract date columns from the products row, starting from column 1
date_row = retail_rice_raw.iloc[products_row, 1:].values
date_columns = [d for d in date_row if pd.notna(d) and str(d).strip() != '']
print(f"\n✓ Found {len(date_columns)} date columns")
print(f"  First 5 dates: {date_columns[:5]}")
print(f"  Last 5 dates: {date_columns[-5:]}")

# Extract brown rice data
brown_rice_product = retail_rice_raw.iloc[brown_row, 0]
brown_rice_prices = retail_rice_raw.iloc[brown_row, 1:len(date_columns)+1].values
print(f"\n✓ Brown rice: {brown_rice_product}")
print(f"  First 5 prices: {brown_rice_prices[:5]}")
print(f"  Number of prices: {len(brown_rice_prices)}")

# Extract white rice data
white_rice_product = retail_rice_raw.iloc[white_row, 0]
white_rice_prices = retail_rice_raw.iloc[white_row, 1:len(date_columns)+1].values
print(f"\n✓ White rice: {white_rice_product}")
print(f"  First 5 prices: {white_rice_prices[:5]}")
print(f"  Number of prices: {len(white_rice_prices)}")

# Verify matching lengths
print(f"\n✓ Data validation:")
print(f"  Dates: {len(date_columns)}")
print(f"  Brown prices: {len(brown_rice_prices)}")
print(f"  White prices: {len(white_rice_prices)}")

# Create dataframes for each rice type
brown_rice_df = pd.DataFrame({
    'date_str': date_columns[:len(brown_rice_prices)],  # Ensure equal length
    'price_cad': brown_rice_prices,
    'product': 'Brown rice (900g)'
})

white_rice_df = pd.DataFrame({
    'date_str': date_columns[:len(white_rice_prices)],  # Ensure equal length
    'price_cad': white_rice_prices,
    'product': 'White rice (2kg)'
})

# Combine both rice types
rice_long_df = pd.concat([brown_rice_df, white_rice_df], ignore_index=True)

print(f"\n✓ Transformed to long format")
print(f"  Brown rice observations: {len(brown_rice_df)}")
print(f"  White rice observations: {len(white_rice_df)}")
print(f"  Total observations: {len(rice_long_df)}")

print(f"\nSample of transformed data:")
print(rice_long_df.head(10))

TRANSFORMING RETAIL RICE DATA: WIDE → LONG FORMAT
Total rows in retail_rice_raw: 8
Total columns: 1

Searching for rice data rows:
Row 0: Monthly average retail prices for selected products: Food 1 2 3 4
Row 1: Frequency: Monthly
Row 2: Table: 18-10-0245-02
Row 3: Release date: 2025-11-05
Row 4: Geography: Canada, Province or territory, Population centre
Row 5: Footnotes:
Row 6: How to cite: Statistics Canada. Table 18-10-0245-02  Monthly average retail prices for selected products: Food
Row 7: https://www150.statcan.gc.ca/t1/tbl1/en/tv.action?pid=1810024502

✓ Found date header row at index 0


IndexError: index 1 is out of bounds for axis 0 with size 1